Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

# Mục mới

In [22]:
!git clone https://github.com/ngoquanhao2009/COVERMUSICNGOQUANHAO.git


fatal: destination path 'COVERMUSICNGOQUANHAO' already exists and is not an empty directory.


In [23]:
import os

# List the contents of the cloned directory
print(os.listdir('COVERMUSICNGOQUANHAO'))

['index.html', 'LICENSE', '.git', 'README.md', 'server', '.gitignore', 'assets', 'requirements.txt']


In [24]:
import os
import re
import uuid
import shutil
import threading
import time
from pathlib import Path
from urllib.parse import urlparse
from flask import Flask, request, jsonify, send_file
from werkzeug.utils import secure_filename
from werkzeug.security import safe_join
from flask_cors import CORS

app = Flask(__name__, static_folder="..", static_url_path="")
CORS(app)

# --- Thêm route để phục vụ index.html cho đường dẫn gốc ---
@app.route('/')
def serve_index():
    # Flask sẽ tìm index.html trong static_folder đã cấu hình
    return app.send_static_file('index.html')

# ---------- Configuration ----------
UPLOAD_FOLDER = Path("/tmp/ai_cover_uploads")
RESULT_FOLDER = Path("/tmp/ai_cover_results")
UPLOAD_FOLDER.mkdir(parents=True, exist_ok=True)
RESULT_FOLDER.mkdir(parents=True, exist_ok=True)

ALLOWED_EXTENSIONS = {"mp3", "wav", "flac", "ogg", "m4a", "aac"}
MAX_CONTENT_LENGTH = 50 * 1024 * 1024  # 50 MB
app.config["MAX_CONTENT_LENGTH"] = MAX_CONTENT_LENGTH

# UUID v4 pattern
_UUID4_RE = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"
)

# Server-side registries – paths are stored here and never reconstructed from user input
uploaded_files: dict[str, dict] = {}   # {file_id: {"path": str, "ext": str}}
jobs: dict[str, dict]           = {}   # {job_id: {..., "result_path": str}}

# ---------- Helpers ----------

def allowed_file(filename: str) -> bool:
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


def sanitize_ext(ext: str) -> str:
    """Return ext only if it is in the allowed set, else raise ValueError."""
    clean = ext.lower().strip()
    if clean not in ALLOWED_EXTENSIONS:
        raise ValueError("Extension not allowed")
    return clean


def validate_uuid(value: str) -> str:
    """Return value if it is a valid UUID v4, else raise ValueError."""
    v = str(value).lower().strip()
    if not _UUID4_RE.match(v):
        raise ValueError("Invalid ID")

def make_upload_path(file_id: str, ext: str) -> Path:
    ext_sane = sanitize_ext(ext)
    file_id_sane = validate_uuid(file_id)
    return UPLOAD_FOLDER / f"{file_id_sane}.{ext_sane}"

def make_result_path(job_id: str, ext: str) -> Path:
    ext_sane = sanitize_ext(ext)
    job_id_sane = validate_uuid(job_id)
    return RESULT_FOLDER / f"{job_id_sane}.{ext_sane}"

def cleanup_file(filepath: str, delay_seconds: int = 300) -> None:
    """Delete a file after a delay in a separate thread."""
    def _delete_file():
        time.sleep(delay_seconds)
        try:
            os.remove(filepath)
            print(f"Cleaned up file: {filepath}")
        except OSError as e:
            print(f"Error cleaning up file {filepath}: {e}")
    threading.Thread(target=_delete_file).start()

def cleanup_folder(folderpath: str, delay_seconds: int = 300) -> None:
    """Delete a folder after a delay in a separate thread."""
    def _delete_folder():
        time.sleep(delay_seconds)
        try:
            shutil.rmtree(folderpath)
            print(f"Cleaned up folder: {folderpath}")
        except OSError as e:
            print(f"Error cleaning up folder {folderpath}: {e}")
    threading.Thread(target=_delete_folder).start()


# ---------- Routes ----------

@app.route("/upload", methods=["POST"])
def upload_file():
    if "file" not in request.files:
        return jsonify({"error": "No file part"}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No selected file"}), 400

    if file and allowed_file(file.filename):
        filename  = secure_filename(file.filename)
        _name, ext = os.path.splitext(filename)
        file_id   = str(uuid.uuid4())
        out_path  = make_upload_path(file_id, ext) # server-generated path
        file.save(str(out_path))
        uploaded_files[file_id] = {"path": str(out_path), "ext": ext}
        return jsonify({"file_id": file_id, "ext": ext, "title": _name, "size": out_path.stat().st_size})
    else:
        return jsonify({"error": "File type not allowed"}), 400


@app.route("/job", methods=["POST"])
def create_job():
    data = request.get_json()
    file_id = data.get("file_id")
    model_params = data.get("model_params")

    if not file_id or not model_params:
        return jsonify({"error": "Missing file_id or model_params"}), 400

    if file_id not in uploaded_files:
        return jsonify({"error": "Invalid file_id"}), 404

    job_id = str(uuid.uuid4())
    # Placeholder for actual model inference
    # In a real application, you would send this to Replicate or a similar service
    result_ext = ".mp3" # Assuming output is always mp3 for now
    result_path = make_result_path(job_id, result_ext) # server-generated path

    # Simulate a job (e.g., copy the input file to result as a placeholder)
    input_path = Path(uploaded_files[file_id]["path"])
    shutil.copy(input_path, result_path)
    # Clean up input file after job is 'created'
    cleanup_file(str(input_path))
    del uploaded_files[file_id]

    jobs[job_id] = {"status": "completed", "result_path": str(result_path), "ext": result_ext}
    return jsonify({"job_id": job_id, "status": "processing"}), 202


@app.route("/job/<job_id>", methods=["GET"])
def get_job_status(job_id: str):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404

    job = jobs[job_id]
    return jsonify({"job_id": job_id, "status": job["status"]}), 200


@app.route("/download/<job_id>", methods=["GET"])
def download_result(job_id: str):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404

    job = jobs[job_id]
    result_path = Path(job["result_path"])

    if not result_path.exists():
        return jsonify({"error": "Result not found or already cleaned up"}), 404

    # Clean up the result file after it's downloaded
    cleanup_file(str(result_path))
    del jobs[job_id]

    return send_file(str(result_path), as_attachment=True, download_name=f"{job_id}{job['ext']}")


@app.route("/youtube", methods=["POST"])
def youtube_download():
    data    = request.get_json()
    raw_url = data.get("url")

    if not raw_url:
        return jsonify({"error": "URL khong duoc de trong"}), 400

    # Validate using URL parser – reject anything not on youtube.com / youtu.be
    try:
        parsed = urlparse(raw_url)
        hostname = parsed.hostname or ""
        # Strip www. / m. prefixes before comparing
        bare = re.sub(r"^(www\.|m\.)", "", hostname)
        if bare not in ("youtube.com", "youtu.be"):
            raise ValueError("Not a YouTube URL")
    except (ValueError, AttributeError):
        return jsonify({"error": "URL khong phai YouTube"}), 400

    try:
        import yt_dlp  # type: ignore
    except ImportError:
        return jsonify({"error": "yt-dlp chua duoc cai. Chay: pip install yt-dlp"}), 501

    file_id  = str(uuid.uuid4())
    out_path = make_upload_path(file_id, "mp3")   # server-generated path
    ydl_opts = {
        "format":         "bestaudio/best",
        "outtmpl":        str(out_path).replace(".mp3", ".%(ext)s"),
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "320"}],
        "quiet":          True,
        "no_warnings":    True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info  = ydl.extract_info(raw_url, download=True)
        title = info.get("title", "YouTube Audio")

    if not out_path.exists():
        return jsonify({"error": "Tai xuong that bai"}), 500

    cleanup_file(str(out_path))
    uploaded_files[file_id] = {"path": str(out_path), "ext": "mp3"}
    return jsonify({"file_id": file_id, "ext": "mp3", "title": title, "size": out_path.stat().st_size})


# ---------- Serve static files ----------

@app.route("/<path:filename>")
def serve_static(filename: str):
    """Serve files from the project root (one level above server/)."""
    root = str(Path("..").resolve())
    # werkzeug.security.safe_join raises NotFound if the path escapes root
    safe = safe_join(root, filename)
    if safe is None or not Path(safe).is_file():
        return "Not found", 404
    return send_file(safe)


if __name__ == "__main__":
    port  = int(os.environ.get("PORT", 5000))
    debug = os.environ.get("FLASK_DEBUG", "false").lower() == "true"
    app.run(host="0.0.0.0", port=port, debug=debug)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/serving.py", line 760, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/http/server.py", line 140, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.12/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3163/1175487462.py", line 236, in <cell line: 0>
    app.run(host="0.0.0.0", port=port, debug=debug)
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 662, in run
    run_simple(t.cast(str, host), port, self, **options)
  File "/usr/local/lib/python3.12/dist-packages/werkz

TypeError: object of type 'NoneType' has no len()

In [ ]:
!pip install -r COVERMUSICNGOQUANHAO/requirements.txt

In [ ]:
import subprocess
import os

# Install ngrok if not already installed
!pip install pyngrok -q

# --- Thêm lệnh để giải phóng cổng 5000 trước khi khởi chạy Flask ---
# Giết bất kỳ tiến trình nào đang sử dụng cổng 5000
!fuser -k 5000/tcp

# Run the Flask app in the background
# Use nohup and & to detach the process from the current shell
flask_command = "python COVERMUSICNGOQUANHAO/server/app.py"
subprocess.Popen(f"nohup {flask_command} > app.log 2>&1 &", shell=True)

print("Flask app đang chạy ở chế độ nền. Đang chờ ngrok khởi tạo...")

# Expose the Flask app to the internet using ngrok
from pyngrok import ngrok

# Terminate any existing ngrok tunnels
ngrok.kill()

# --- THAY THẾ "YOUR_NGROK_AUTH_TOKEN" BẰNG MÃ AUTH TOKEN CỦA BẠN TẠI ĐÂY ---
# Bạn có thể lấy mã này từ trang dashboard của ngrok sau khi đăng ký tài khoản:
# https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3CRbdOwCYVhHOGpOvcJMRaM90mt_3zeFWq1cEaPrEo4VAdnUd") # <-- Thay thế placeholder này bằng authtoken của bạn

# Start a new ngrok tunnel for port 5000 (default Flask port)
public_url = ngrok.connect(5000)
print(f"Flask App Public URL: {public_url}")

Bây giờ, hãy kiểm tra xem ứng dụng Flask của bạn có hoạt động qua đường dẫn `ngrok` hay không bằng cách gửi một yêu cầu HTTP đơn giản đến URL công khai.

In [ ]:
import requests

# Lấy public_url từ biến đã được tạo ở cell trước
# Nếu bạn đã chạy lại Colab, có thể cần lấy lại giá trị public_url
# hoặc đảm bảo rằng biến public_url đã được định nghĩa.

# Nếu biến public_url không tồn tại, bạn có thể thay thế bằng URL của mình:
# public_url_str = "https://crispy-drab-debtor.ngrok-free.dev"
# Hoặc đảm bảo cell trước đã chạy để có biến `public_url`

if 'public_url' in locals():
    # Lấy URL thực tế từ thuộc tính public_url của đối tượng NgrokTunnel
    public_url_str = public_url.public_url
else:
    print("Lỗi: Không tìm thấy biến `public_url`. Vui lòng chạy lại cell ngrok.")
    public_url_str = ""

if public_url_str:
    try:
        # Gửi yêu cầu GET đến URL gốc của ứng dụng Flask
        response = requests.get(public_url_str)

        print(f"Trạng thái phản hồi: {response.status_code}")
        print("Nội dung phản hồi (100 ký tự đầu tiên):")
        print(response.text[:500]) # In 500 ký tự đầu tiên của phản hồi

        if response.status_code == 200:
            print("Ứng dụng Flask đang hoạt động và có thể truy cập qua ngrok!")
        else:
            print("Ứng dụng Flask không trả về mã trạng thái 200. Có thể có vấn đề.")

    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi gửi yêu cầu HTTP: {e}")
        print("Đảm bảo rằng ứng dụng Flask đang chạy và URL ngrok là chính xác.")
else:
    print("Không thể kiểm tra vì `public_url` không hợp lệ.")

Ứng dụng Flask đang trả về lỗi 500 Internal Server Error. Để tìm hiểu nguyên nhân, chúng ta cần xem log của ứng dụng.

In [ ]:
# Đọc nội dung của file log để kiểm tra lỗi
log_file_path = 'app.log'

try:
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
except FileNotFoundError:
    print(f"Lỗi: File log '{log_file_path}' không tìm thấy.")
except Exception as e:
    print(f"Có lỗi xảy ra khi đọc file log: {e}")

In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

In [ ]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)